In [1]:
import numpy as np
# import matplotlib.pyplot as plt
# import ipywidgets as iw
# from numpy.random import uniform as nprand_unif

In [ ]:
nV = 16 # how many vertices
p0 = 0.3 # p of a given vertex being type0 (for initial conditions only)
pE = 0.6 # p of a given edge being ON (for initial conditions only)

initV = nprand_unif(size=(nV,)) < p0

_ = nprand_unif(size=(nV,nV)) < pE**(1/2)
initE = _ * np.transpose(_) # logical_and
np.fill_diagonal(initE, False)

In [ ]:
T_alloc = 10
V = np.zeros((T_alloc, nV), np.bool)
E = np.zeros((T_alloc, nV, nV), np.bool)

V[0,:] = initV
E[0,:,:] = initE
T_known = 1

In [ ]:
def compute_next():
    global T_known
    T_inprogress = T_known + 1

    if T_inprogress > T_alloc:
        return# handle the out-of-array-space condition

    # mutate arrays by assigning to slice
    # all the logic comes before here 
    V[T_inprogress-1] = ~V[T_known-1]
    E[T_inprogress-1] = ~E[T_known-1]

    T_known = T_inprogress

In [ ]:
out = iw.Output()

# Fixed node positions (circle)
theta = np.linspace(0, 2*np.pi, nV, endpoint=False)
pos = np.c_[np.cos(theta), np.sin(theta)]

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig) # prevent duplicate display
with out:
    display(fig)

def draw_state(t):
    print(f'Drawing from V[{t-1}]')
    ax.clear()

    ON_nodes = V[t-1]
    print(pos[ON_nodes], pos[~ON_nodes])
    ax.scatter(pos[~ON_nodes, 0], pos[~ON_nodes, 1], c='gray', s=50)
    ax.scatter(pos[ON_nodes, 0], pos[ON_nodes, 1], c='red', s=80)

    edges = np.argwhere(E[t-1])
    for i, j in edges:
        if i < j:
            ax.plot(
                [pos[i, 0], pos[j, 0]],
                [pos[i, 1], pos[j, 1]],
                c='black',
                alpha=0.5
            )

    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f"State {t}")
    fig.canvas.draw_idle()

# -----------------------------
# Stats panel
# -----------------------------
stats = iw.HTML()

def update_stats(t):
    num_on = V[t-1].sum()
    num_edges = E[t-1].sum() // 2
    avg_deg = 2 * num_edges / nV

    stats.value = (
        f"<b>ON nodes:</b> {num_on}<br>"
        f"<b>Total edges:</b> {num_edges}<br>"
        f"<b>Avg degree:</b> {avg_deg:.2f}"
    )


In [ ]:
T_displayed = 1

def on_slider_change(change):
    global T_displayed
    T_slidertarget = change['new']
    T_displayed = T_slidertarget

    with out:
        draw_state(T_slidertarget)
    update_stats(T_slidertarget)

slider = iw.IntSlider(
    value=1, min=1, max=1, step=1,
    description='t', continuous_update=False
)
slider.observe(on_slider_change, names='value')

In [ ]:
def on_compute_next_clicked(b):
    compute_next()
    # update slider range, then go to latest
    slider.max = T_known - 1
    slider.value = slider.max

compute_display_next_btn = iw.Button(description="Compute & display next")
compute_display_next_btn.on_click(on_compute_next_clicked)

In [ ]:
with out:
    draw_state(1)
update_stats(1)

display(
    iw.VBox([
        iw.HBox([
            slider, compute_display_next_btn
        ]),
        out,
        stats
    ])
)